In [1]:
%pip install sentence-transformers

  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/780.3 kB ? eta -:--:--
   ------------- -------------------------- 262.1/780.3 kB ? eta -:--:--
   ---------------------------------------- 780.3/780.3 kB 3.4 MB/s  0:00:00
   ---------------------------------------- 0.0/857.0 kB ? eta -:--:--
   ---------------------------------------- 857.0/857.0 kB 6.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------- -------------------------- 1.3/3.9 MB 7.4 MB/s eta 0:00:01
   ----------------------------- ---------- 2.9/3.9 MB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 7.2 MB/s  0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.6/13.1 MB 7.9 MB/s eta 0:00:02
   --------- ------------------------------ 3.1/13.1 MB 8.2 MB/s eta 0:00:02
   -------------- ------------------------


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\lenovo\Desktop\AI Learning\job-match-assistant\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded


In [2]:
sentences = [
    "Machine learning engineer",
    "AI developer building ML models",
    "Sales manager for retail stores",
]

embeddings = model.encode(sentences)
print(embeddings.shape)
print(embeddings[0][:5])

(3, 384)
[-0.07065053 -0.03912071  0.05264389  0.04696643 -0.01286515]


In [3]:
from sentence_transformers import util

scores = util.cos_sim(embeddings, embeddings)
print(scores)

tensor([[1.0000, 0.5574, 0.2309],
        [0.5574, 1.0000, 0.0644],
        [0.2309, 0.0644, 1.0000]])


In [4]:
import pandas as pd

df = pd.read_csv("../data/jobs.csv")
sample = df.dropna(subset=["title", "description"]).sample(2000, random_state=42).reset_index(drop=True)

print(sample.shape)
print(sample[["title"]].head())

(2000, 31)
                                               title
0                                          Warehouse
1                 Influencer Talent Manager - Beauty
2                      Talent Attraction Specialist 
3           Retail Sales - Shoes - Stonebriar Centre
4  Physician Assistant, Per Diem - Internal Medicine


In [5]:
import numpy as np

texts = (sample["title"] + ". " + sample["description"].str[:1000]).tolist()

job_embeddings = model.encode(texts, show_progress_bar=True)
np.save("../data/job_embeddings.npy", job_embeddings)

print(job_embeddings.shape)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

(2000, 384)


In [6]:
resume = """
Computer Science graduate with practical training and project-based experience
in Data Analytics, Python programming, SQL, and Business Intelligence.

Education:
Bachelor of Science in Computer Science (BSCS).
IBM Data Analyst Professional Certificate - 11 courses completed.
NAVTTC Data Analytics Blended Programme.

Technical Skills:
Python, SQL, PostgreSQL, Excel, Advanced Excel, Google Sheets,
Power BI, Tableau, pandas, NumPy, Matplotlib, Seaborn,
Jupyter Notebook, Data Cleaning, Data Transformation,
Exploratory Data Analysis (EDA), Data Visualization,
Dashboard Development, Data Reporting, Data Modeling,
Data Interpretation, Statistical Analysis, Git and GitHub.

Practical Projects:
1. Urban Basket Retail Sales Analysis:
Analyzed retail sales data using PostgreSQL, Python, and Power BI.
Developed business performance measures, sales KPIs,
customer analysis, and interactive dashboards.

MetroCare Hospital Analytics:
Performed data cleaning and transformation using Power Query.
Analyzed hospital service revenue, patient records,
appointments, and billing data.
Developed a Power BI dashboard.

SkyLink Airlines SQL Analysis:
Performed SQL-based analysis of airline bookings,
revenue, cancellations, and business performance.

Career Objective:
Seeking entry-level opportunities as a Data Analyst,
Junior Data Analyst, Business Intelligence Analyst,
SQL Analyst, or Python Data Analyst.

Interested in transforming raw data into meaningful
business insights and supporting data-driven decisions.
"""

resume_embedding = model.encode(resume)

scores = util.cos_sim(resume_embedding, job_embeddings)[0]

top5 = scores.argsort(descending=True)[:5]

for i in top5:
    print(round(float(scores[i]), 3), "|", sample.loc[int(i), "title"])

0.624 | Business Data Analyst
0.584 | Data Analyst-WA
0.58 | Financial Analyst
0.576 | Healthcare Economics Analyst
0.57 | Data Engineer


In [7]:
print("Model limit:", model.max_seq_length)
print("Resume length:", len(model.tokenizer(resume)["input_ids"]))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (297 > 256). Running this sequence through the model will result in indexing errors


Model limit: 256
Resume length: 297


In [8]:
chunks = [c.strip() for c in resume.split("\n\n") if c.strip()]
print("Number of chunks:", len(chunks))
print("Tokens per chunk:", [len(model.tokenizer(c)["input_ids"]) for c in chunks])

chunk_embeddings = model.encode(chunks)
resume_embedding = chunk_embeddings.mean(axis=0)

scores = util.cos_sim(resume_embedding, job_embeddings)[0]
top5 = scores.argsort(descending=True)[:5]
for i in top5:
    print(round(float(scores[i]), 3), "|", sample.loc[int(i), "title"])

Number of chunks: 8
Tokens per chunk: [26, 34, 92, 47, 36, 27, 31, 18]
0.71 | Business Data Analyst
0.673 | Data Analyst-WA
0.638 | Financial Analyst
0.634 | Data Analyst
0.623 | Data Scientist II


In [9]:
top10 = scores.argsort(descending=True)[:10]
for i in top10:
    i = int(i)
    print(round(float(scores[i]), 3), "|", sample.loc[i, "title"], "|", sample.loc[i, "location"])

best = int(top10[0])
print("\n--- Best match description ---")
print(sample.loc[best, "description"][:600])

0.71 | Business Data Analyst | Rosemont, IL
0.673 | Data Analyst-WA | Washington, United States
0.638 | Financial Analyst | Dublin, OH
0.634 | Data Analyst | Roswell, GA
0.623 | Data Scientist II | Orlando, FL
0.62 | Data Engineer | United States
0.619 | Tableau Architect | Sunnyvale, CA
0.611 | Data Analyst (Korean) | United States
0.607 | Healthcare Economics Analyst | Meridian, ID
0.602 | Business Analyst | Michigan, United States

--- Best match description ---
JOB DESCRIPTION: Business Data Analyst 

Location: Hybrid – Rosemont, IL

Overview: Supports and contributes to business analytics projects and processes to provide data-driven insights related to business performance used to advise and develop strategies for operational improvements and future business initiatives. Uses statistical methods, modeling, analytical methodologies, and data analysis to develop and deploy tools including dashboards, infographics, reports, and models to inform and support decision-making. Mines, pr

In [10]:
def match_resume(resume_text, top_n=5):
    chunks = [c.strip() for c in resume_text.split("\n\n") if c.strip()]
    resume_vec = model.encode(chunks).mean(axis=0)
    scores = util.cos_sim(resume_vec, job_embeddings)[0]
    top = scores.argsort(descending=True)[:top_n]
    return [(round(float(scores[int(i)]), 3), sample.loc[int(i), "title"]) for i in top]

for score, title in match_resume(resume, top_n=10):
    print(score, "|", title)

0.71 | Business Data Analyst
0.673 | Data Analyst-WA
0.638 | Financial Analyst
0.634 | Data Analyst
0.623 | Data Scientist II
0.62 | Data Engineer
0.619 | Tableau Architect
0.611 | Data Analyst (Korean)
0.607 | Healthcare Economics Analyst
0.602 | Business Analyst


## Stage 3 Findings: Text Embeddings and Semantic Matching

- Used the `all-MiniLM-L6-v2` sentence-transformer model to turn text into 384-dimensional vectors that capture meaning.
- Cosine similarity ranked "Machine learning engineer" and "AI developer building ML models" (0.56) far above an unrelated sales role (0.06 to 0.23), something TF-IDF from Stage 2 could not do.
- Embedded a random sample of 2,000 LinkedIn postings (title + first 1,000 characters of description) and saved them to `data/job_embeddings.npy`.
- The resume was 297 tokens, over the model's 256-token limit, so the end was being cut off. Splitting it into 8 chunks, embedding each, and averaging the vectors fixed this.
- Top matches for my resume: Business Data Analyst (0.71), Data Analyst-WA (0.67), Financial Analyst (0.64), Data Analyst (0.63), Data Scientist II (0.62). All are relevant analyst roles.
- Wrapped the process in a reusable `match_resume()` function.
- Limitation: the dataset is mostly US postings, so local Rawalpindi/Islamabad jobs need a different dataset.